In [ ]:
import asyncio
from websockets.asyncio.server import serve
import random
import socket

In [ ]:
paddleSpeed = "5"
ids = []
players = {}
#{
#    # id : [corY, corX]
#}

x = 600 // 2
y = 600 // 2
ball = str(x) + "," + str(y)
speedX = random.uniform(-3, 5)
speedY = random.uniform(-3, 5)
WIDTH, HEIGHT = 600, 600
rectSize = 75
ip = socket.gethostbyname(socket.gethostname())

In [ ]:

async def respond(websocket):
    global x,y,speedX, speedY
    async for message in websocket:
        messages = message.split(",")
        opcode = messages[0]
        response = ""
        if opcode == "getId":
            playerId = str(hash(random.uniform(0,10)))
            if len(list(players.keys())) > 0:
                
                x = WIDTH - 30
            else:
                x = 20
            players[playerId] = [messages[1], str(x)]
            response = playerId + "," + players[playerId][0] + "," + players[playerId][1]
        elif opcode == "getOpp":
            opId = str(messages[1]) 
            response = players[opId][0] + "," + players[opId][1]
        elif opcode == "getIds":
            response = ""
            ids = list(players.keys())
            for i in range(len(ids)):
                response += ids[i-1]
                response += ","
            response = response[:-1]
        elif opcode == "setCoor":
            id = messages[1]
            players[id][0] = messages[2]
            response = players[id][0] + "," + players[id][1]
        elif opcode == "getBall":
            playerids = list(players.keys())
            player = players[playerids[0]]
            playerbtn = float(player[0]) - rectSize / 2
            playertop = float(player[0]) + rectSize / 2
            playerx = float(player[1])
            opp = players[playerids[1]]
            oppbtn = float(opp[0]) - rectSize / 2
            opptop = float(opp[0]) + rectSize / 2
            oppx = float(opp[1])
            if playerx < x < playerx + 10 and playerbtn < y < playertop:
                speedX *= -1 
            if oppx < x <  oppx + 10 and oppbtn < y < opptop:
                speedX *= -1
            if y < 0 or y > HEIGHT:
                speedY *= -1
            x += speedX
            y += speedY
            response = str(x) + "," + str(y)
        elif opcode == "resetBall":
            x = 600 // 2
            y = 600 // 2
            speedX = random.uniform(-3, 5)
            speedY = random.uniform(-3, 5)
            response = str(x) + "," + str(y)

        await websocket.send(response)

In [ ]:

async def main():
    print("Server started on ws://{}:8765".format(ip))
    async with serve(respond, ip, 8765) as server:
        await server.serve_forever()


In [ ]:

asyncio.run(main())